# Imports

In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/ncdc-lassa-fever-timeseries-20202025/lassa_fever_timeseries_minimal.csv
/kaggle/input/ncdc-lassa-fever-timeseries-20202025/lassa_fever_timeseries_full.csv
/kaggle/input/4-classical-models/metrics_table.csv
/kaggle/input/4-classical-models/__results__.html
/kaggle/input/4-classical-models/lb_ets_resid.csv
/kaggle/input/4-classical-models/__notebook__.ipynb
/kaggle/input/4-classical-models/__output__.json
/kaggle/input/4-classical-models/lb_sarima_resid.csv
/kaggle/input/4-classical-models/custom.css
/kaggle/input/4-classical-models/__results___files/__results___13_3.png
/kaggle/input/4-classical-models/__results___files/__results___16_2.png
/kaggle/input/4-classical-models/__results___files/__results___13_2.png
/kaggle/input/4-classical-models/__results___files/__results___16_3.png
/kaggle/input/4-classical-models/__results___files/__results___13_1.png
/kaggle/input/4-classical-models/__results___files/__results___16_1.png


# Functions

In [2]:
# Mean Absolute Error (MAE)
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

# Root Mean Squared Error (RMSE)
def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

# Mean Absolute Percentage Error (MAPE)
def mape(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0  # avoid division by zero
    return (np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100)

# Function to calculate all three metrics
def calculate_evaluation_metrics(y_true, y_pred):
    metrics = []
    metrics.append(mae(y_true, y_pred).round(2))
    metrics.append(rmse(y_true, y_pred).round(2))
    metrics.append(mape(y_true, y_pred).round(2))
    return metrics


# plot graph 
def plot_forecasts(forecast, title):
    """Function to plot the forecasts"""
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=train.index, y=train['confirmed_cases'], name="Train", line=dict(color='#1f77b4')))
    fig.add_trace(go.Scatter(x=test.index, y=test['confirmed_cases'], name="Test", line=dict(color='#2ca02c')))
    fig.add_trace(go.Scatter(x=test.index, y=forecast, name="Forecast", line=dict(color='#ff0000')))
    fig.update_layout(template="simple_white", font=dict(size=18), title_text=title,
                     width=650, title_x=0.5, height=400, xaxis_title='Date',
                     yaxis_title='Confirmed Cases')
    return fig.show()


metrics_table = pd.read_csv('/kaggle/input/4-classical-models/metrics_table.csv')

# Data Extraction and EDA

In [3]:
# read data into a dataframe
df = pd.read_csv('/kaggle/input/ncdc-lassa-fever-timeseries-20202025/lassa_fever_timeseries_minimal.csv')

# construct series dataframe
ts_data = df[['week_end_date', 'epi_week', 'confirmed_cases']].copy()
ts_data.columns = ['date', 'epi_week', 'confirmed_cases']

# Ensure datetime index
ts_data['date'] = pd.to_datetime(ts_data['date'])
ts_data = ts_data.set_index('date')
ts_data = ts_data.asfreq('W') # set frequency to weekly
ts_data

,epi_week,confirmed_cases
date,,
2020-01-05,1,18
2020-01-12,2,64
2020-01-19,3,81
2020-01-26,4,95
2020-02-02,5,104
...,...,...
2025-10-19,42,9
2025-10-26,43,11
2025-11-02,44,12


In [4]:
# Some basic EDA
print(ts_data.dtypes)
print(ts_data.isnull().sum())
print(ts_data.describe())

epi_week           int64
confirmed_cases    int64
dtype: object
epi_week           0
confirmed_cases    0
dtype: int64
         epi_week  confirmed_cases
count  307.000000       307.000000
mean    26.136808        20.540717
std     14.879500        25.897753
min      1.000000         0.000000
25%     13.000000         6.000000
50%     26.000000        10.000000
75%     39.000000        21.000000
max     53.000000       137.000000


In [5]:
# create and display plot 
fig =  px.line(ts_data, x=ts_data.index, y="confirmed_cases",
        title="Weekly Lassa Fever Cases (2020–2025)")
fig.show()

In [6]:
# create time lags 
ts_data['lag_1'] = ts_data['confirmed_cases'].shift(1)
ts_data['lag_2'] = ts_data['confirmed_cases'].shift(2)
ts_data['lag_3'] = ts_data['confirmed_cases'].shift(3)
ts_data['lag_4'] = ts_data['confirmed_cases'].shift(4)

In [7]:
# drop missing rows
ts_data = ts_data.dropna()

In [8]:
ts_data

,epi_week,confirmed_cases,lag_1,lag_2,lag_3,lag_4
date,,,,,,
2020-02-02,5,104,95.0,81.0,64.0,18.0
2020-02-09,6,109,104.0,95.0,81.0,64.0
2020-02-16,7,115,109.0,104.0,95.0,81.0
2020-02-23,8,102,115.0,109.0,104.0,95.0
2020-03-01,9,85,102.0,115.0,109.0,104.0
...,...,...,...,...,...,...
2025-10-19,42,9,11.0,13.0,4.0,11.0
2025-10-26,43,11,9.0,11.0,13.0,4.0
2025-11-02,44,12,11.0,9.0,11.0,13.0


# Train Test Split

In [9]:
# Train Test Split
train = ts_data.iloc[:-int(len(ts_data)*0.2)].copy()
test = ts_data.iloc[-int(len(ts_data)*0.2):].copy()

# XGB Regression

In [10]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
)

X_train = train.drop('confirmed_cases', axis=1)
y_train = train['confirmed_cases']

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=5, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=300, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [11]:
X_test = test.drop('confirmed_cases', axis=1)
y_pred = model.predict(X_test)

In [12]:
# Plot the forecasts
plot_forecasts(y_pred, 'XGB Forecast')

# Calculate metrics and add to results dataframe
metrics_list = ['XGB'] + calculate_evaluation_metrics(test['confirmed_cases'], y_pred)
metrics_table.loc[len(metrics_table)] = metrics_list

In [13]:
metrics_table

,Model,MAE,RMSE,MAPE
0,Naive,16.33,26.37,60.30
1,Average,16.20,20.94,129.03
2,Drift,17.69,27.12,69.84
3,Seasonal Naive,8.74,15.53,49.24
4,Seasonal Mean,7.91,12.25,44.11
5,ETS,8.06,12.79,43.90
6,SARIMA,7.64,12.45,44.46
7,XGB,7.68,11.94,42.76
